**Mount Drive**

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Install YOLO**

In [5]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.5 MB/s eta 0:00:00


**Import**

In [6]:
from ultralytics import YOLO
import os
import shutil

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [7]:
path_disease = '/content/drive/MyDrive/Colab Notebooks/Chilli Plant Dataset/Chili Leaf Disease Original Dataset'
path_growth = '/content/drive/MyDrive/Colab Notebooks/Chilli Plant Dataset/Chili Growth Stage Original Dataset'

In [8]:
from PIL import Image

def convert_to_yolo(input_path, output_path):
    classes = sorted(os.listdir(input_path))
    class_map = {cls:i for i, cls in enumerate(classes)}

    os.makedirs(f"{output_path}/images/train", exist_ok=True)
    os.makedirs(f"{output_path}/labels/train", exist_ok=True)

    for cls in classes:
        cls_path = os.path.join(input_path, cls)

        for img_name in os.listdir(cls_path):
            if not img_name.lower().endswith(('.jpg','.png','.jpeg')):
                continue

            src = os.path.join(cls_path, img_name)
            dst = os.path.join(output_path, "images/train", img_name)

            shutil.copy(src, dst)

            label_path = os.path.join(output_path, "labels/train",
                                      img_name.replace('.jpg','.txt').replace('.png','.txt'))

            with open(label_path, "w") as f:
                # FULL IMAGE BOX (classification style)
                f.write(f"{class_map[cls]} 0.5 0.5 1.0 1.0")

    return class_map

In [9]:
disease_output = "/content/disease_yolo"
growth_output = "/content/growth_yolo"

disease_classes = convert_to_yolo(path_disease, disease_output)
growth_classes = convert_to_yolo(path_growth, growth_output)

In [10]:
def create_yaml(output_path, class_map, file_name):
    yaml_text = f"""
path: {output_path}
train: images/train
val: images/train

names:
"""
    for name, idx in class_map.items():
        yaml_text += f"  {idx}: {name}\n"

    with open(file_name, "w") as f:
        f.write(yaml_text)

In [11]:
create_yaml(disease_output, disease_classes, "disease.yaml")
create_yaml(growth_output, growth_classes, "growth.yaml")

**LOAD YOLO MODELS**

In [12]:
model_disease = YOLO("yolov8n.pt")
model_growth = YOLO("yolov8n.pt")

**TRAIN**

In [ ]:
print("--- Training Disease Model ---")
model_disease.train(data="disease.yaml", epochs=10)

print("\n--- Training Growth Model ---")
model_growth.train(data="growth.yaml", epochs=10)

--- Training Disease Model ---
Ultralytics 8.4.31 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=disease.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:839: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/10         0G          0       80.6          0          0        640: 100% ━━━━━━━━━━━━ 116/116 11.2s/it 21:35
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 58/58 9.8s/it 9:28
                   all       1856          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels


/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:839: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/10         0G          0      57.74          0          0        640: 100% ━━━━━━━━━━━━ 116/116 11.1s/it 21:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 14% ━╸────────── 8/58 9.9s/it 1:16<8:16

In [ ]:
save_path = '/content/drive/MyDrive/Colab Notebooks/Chilli Plant Dataset/Saved_Models'
os.makedirs(save_path, exist_ok=True)

shutil.copy("runs/detect/train/weights/best.pt",
            f"{save_path}/chili_disease_yolo.pt")

print("Disease YOLO model saved")

In [ ]:
shutil.copy("runs/detect/train2/weights/best.pt",
            f"{save_path}/chili_growth_yolo.pt")

print("Growth YOLO model saved")

In [ ]:
model_disease = YOLO(f"{save_path}/chili_disease_yolo.pt")
model_growth = YOLO(f"{save_path}/chili_growth_yolo.pt")

In [ ]:
def final_predict(img_path):
    print("-" * 40)
    print(f"FILE: {os.path.basename(img_path)}")

    d_results = model_disease(img_path)
    g_results = model_growth(img_path)

    for r in d_results:
        for box in r.boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            print(f"DISEASE: {model_disease.names[cls]} ({conf*100:.2f}%)")

    for r in g_results:
        for box in r.boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            print(f"GROWTH: {model_growth.names[cls]} ({conf*100:.2f}%)")

    print("-" * 40)

In [ ]:
test_image_path = "/content/Screenshot 2026-03-29 004942.png"
final_predict(test_image_path)